In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Section 4.2 — ML in Finance

**Objectives**

By the end of this section you will be able to:

- Build a credit scoring model using Gradient Boosting.
- Evaluate a financial risk model with AUC-ROC and a classification report.
- Apply Isolation Forest for unsupervised fraud and anomaly detection.
- Discuss ethical considerations in algorithmic financial decision-making.

> **Cooking analogy:** A finance team is like a **head chef managing a large
> restaurant kitchen**: they must decide which suppliers (borrowers) are reliable,
> flag any ingredient (transaction) that smells off before it contaminates the
> whole batch, and balance the menu (portfolio) so no single dish dominates. ML
> automates these quality-control decisions at scale and with far greater
> consistency than manual inspection alone.

Finance teams use ML for credit scoring, fraud detection, portfolio management,
and risk assessment.

### Credit Scoring Model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
n = 2000

df_credit = pd.DataFrame({
    "Age"              : np.random.randint(18, 70, n),
    "Annual_Income"    : np.random.normal(55_000, 20_000, n).clip(15_000, 200_000).round(-2),
    "Credit_History"   : np.random.randint(0, 20, n),         # years
    "Existing_Debt"    : np.random.normal(15_000, 10_000, n).clip(0, 80_000).round(-2),
    "Employment_Status": np.random.choice([0, 1], n, p=[0.2, 0.8]),  # 0=unemployed
    "Num_Loans"        : np.random.randint(0, 8, n),
})

# Default probability influenced by debt ratio and employment
debt_ratio       = df_credit["Existing_Debt"] / df_credit["Annual_Income"]
default_prob     = (0.3 * debt_ratio + 0.2 * (1 - df_credit["Employment_Status"])
                    + 0.1 * (df_credit["Num_Loans"] / 8)).clip(0, 1)
df_credit["Default"] = np.random.binomial(1, default_prob)

print("Default rate: {:.1%}".format(df_credit["Default"].mean()))

X = df_credit.drop(columns="Default")
y = df_credit["Default"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

gb_model = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
gb_model.fit(X_tr_sc, y_tr)

y_prob_credit = gb_model.predict_proba(X_te_sc)[:, 1]
print(f"\nAUC-ROC: {roc_auc_score(y_te, y_prob_credit):.3f}")
print(classification_report(y_te, gb_model.predict(X_te_sc),
                             target_names=["No Default","Default"]))

In [ ]:
# Assign credit scores (higher = lower default risk)
all_prob = gb_model.predict_proba(scaler.transform(X))[:, 1]
df_credit["Credit_Score"] = (1000 * (1 - all_prob)).round(0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df_credit["Credit_Score"], bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Predicted Credit Score Distribution")
axes[0].set_xlabel("Score")
axes[0].set_ylabel("Count")

RocCurveDisplay.from_estimator(gb_model, X_te_sc, y_te, ax=axes[1],
                                name="Gradient Boosting")
axes[1].set_title("ROC Curve — Default Prediction")

plt.tight_layout()
plt.show()

### Fraud Detection

In [ ]:
from sklearn.ensemble import IsolationForest

np.random.seed(55)
n_normal = 1900
n_fraud  = 100

df_fraud = pd.DataFrame({
    "Amount"    : np.concatenate([np.random.lognormal(4, 1, n_normal),
                                   np.random.uniform(5_000, 20_000, n_fraud)]),
    "Hour"      : np.concatenate([np.random.randint(6, 22, n_normal),
                                   np.random.randint(0, 6, n_fraud)]),
    "Merchant_Risk": np.concatenate([np.random.uniform(0, 0.3, n_normal),
                                       np.random.uniform(0.7, 1.0, n_fraud)]),
    "True_Fraud": [0] * n_normal + [1] * n_fraud
})

X_fraud = df_fraud[["Amount","Hour","Merchant_Risk"]]
iso     = IsolationForest(contamination=0.05, random_state=42)
iso.fit(X_fraud)
df_fraud["Predicted_Fraud"] = (iso.predict(X_fraud) == -1).astype(int)

tp = ((df_fraud["True_Fraud"] == 1) & (df_fraud["Predicted_Fraud"] == 1)).sum()
print(f"Fraud correctly flagged (Recall): {tp/n_fraud:.1%}")

---

### Student Task 4.2

1. Using `df_credit`, plot **feature importance** for the Gradient Boosting model. Which three factors most strongly predict loan default?
2. Create a bar chart showing **average default rate by number of existing loans** (`Num_Loans`). What pattern emerges?
3. What **ethical concerns** should a bank consider when using an ML credit scoring model? Write 2–3 sentences.

In [ ]:
# Your code here

---

### Evaluation Questions 4.2

1. In credit scoring, a **high AUC-ROC** score indicates:
   a) The model makes many false positives
   b) The model is good at distinguishing defaulters from non-defaulters (correct)
   c) The loan approval rate is high
   d) The model was trained on a large dataset

2. **Gradient Boosting** builds models by:
   a) Training one large decision tree
   b) Averaging many independent trees in parallel
   c) Sequentially adding trees that correct previous errors (correct)
   d) Clustering customers before prediction

3. An Isolation Forest detects anomalies by:
   a) Calculating the distance from cluster centroids
   b) Identifying points that are easy to isolate in fewer splits (correct)
   c) Using logistic regression probabilities
   d) Training on only fraudulent transactions

4. Why is **class imbalance** a challenge in fraud detection?
   a) Fraud happens too frequently
   b) The model may learn to predict "no fraud" for all cases and still get high accuracy (correct)
   c) There are too many features in financial datasets
   d) Neural networks cannot detect fraud

5. What does `predict_proba()` return for a binary classifier?
   a) The predicted class label (0 or 1)
   b) The feature importance array
   c) A probability for each class (correct)
   d) The confusion matrix

---